<a href="https://colab.research.google.com/github/hiroaki-com/colab-mosaic-clip/blob/main/mosaic_clip_ja.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Colab Moza Clip

動画の任意の時間範囲・位置に、モザイクまたは単色マスクをかけて MP4 で出力する Colab ノートブックです。無料かつ簡単・安全にご使用いただけます。

| ステップ | 概要 |
|:---:|---|
| 1. | セットアップ（自動） |
| 2. | 動画アップロード → 開始/終了秒を指定 → クリックでモザイク位置を指定 |
| 3. | 保存先を選んで書き出し（MP4形式） |
| 4. | ファイルを`ローカル`または`Google Drive`に保存 |


In [ ]:
# @title 1. セットアップ { display-mode: "form" }
# @markdown ffmpeg・Pillow のインストールと、共通ユーティリティの定義を行います。
# @markdown
# @markdown 最初に一度だけ実行してください。

!apt-get install -y ffmpeg -q
!pip install pillow -q

import os, subprocess, base64, shutil, threading, json
import ipywidgets as w
from IPython.display import display, Image, Video, HTML
from google.colab import files as colab_files, output as colab_output, drive
from PIL import Image as PILImage, ImageDraw
from io import BytesIO

_PREVIEW_JPEG_QUALITY = 85
_MIN_MOSAIC_BLOCK_PX  = 8
_MIN_SELECTION_SIZE   = 10
_STDERR_DRAIN_TIMEOUT = 2

def _safe_float(s, default=0.0):
    try:
        return float(s)
    except (ValueError, TypeError):
        return default

def _ffmpeg_progress(args, label='処理'):
    cmd = ['ffmpeg', '-y', '-loglevel', 'error', '-progress', 'pipe:1'] + [str(a) for a in args]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    fps_v, frame_v, speed_v = None, None, None
    stderr_lines = []
    def _drain_stderr():
        for line in proc.stderr:
            stderr_lines.append(line)
    t = threading.Thread(target=_drain_stderr, daemon=True)
    t.start()
    for line in proc.stdout:
        line = line.strip()
        if line.startswith('fps='):    fps_v   = line.split('=')[1]
        if line.startswith('frame='): frame_v  = line.split('=')[1]
        if line.startswith('speed='): speed_v  = line.split('=')[1]
        if line == 'progress=continue' and frame_v:
            parts = [f'フレーム: {frame_v}']
            if fps_v and fps_v != '0': parts.append(f'FPS: {fps_v}')
            if speed_v:                parts.append(f'速度: {speed_v}')
            print(f'   ⏳ {label} — ' + ' / '.join(parts), end='\r')
    proc.wait()
    t.join(timeout=_STDERR_DRAIN_TIMEOUT)
    print()
    if proc.returncode != 0:
        print(f'❌ ffmpeg エラー:\n' + ''.join(stderr_lines)[-2000:])
        raise RuntimeError(f'ffmpeg failed (returncode={proc.returncode})')

def _ffmpeg(*args):
    cmd = ['ffmpeg', '-y', '-loglevel', 'warning'] + [str(a) for a in args]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        print('❌ ffmpeg エラー:\n' + proc.stderr[-2000:])
        raise RuntimeError(f'ffmpeg failed (returncode={proc.returncode})')

def _probe(path, entry):
    res = subprocess.run(
        ['ffprobe', '-v', 'error', '-show_entries', f'format={entry}',
         '-of', 'default=noprint_wrappers=1:nokey=1', path],
        capture_output=True, text=True
    )
    return res.stdout.strip()

def _build_filter_complex(selections, t_start, t_end, orig_w=0, orig_h=0, out='[vout]'):
    crops, overlays = [], []
    prev = '[0:v]'
    for i, sel in enumerate(selections):
        x0 = max(0, sel['x'] - sel['w'] // 2)
        y0 = max(0, sel['y'] - sel['h'] // 2)
        cw = (sel['x'] + sel['w'] // 2) - x0
        ch = (sel['y'] + sel['h'] // 2) - y0
        if orig_w > 0:
            cw = min(cw, orig_w - x0)
            ch = min(ch, orig_h - y0)
        label = out if i == len(selections) - 1 else f'[tmp{i}]'
        if sel.get('pattern') == 'solid':
            clr = sel.get('color', '#808080').lstrip('#')
            crops.append(f'color=c=#{clr}:size={cw}x{ch}[mz{i}]')
        else:
            block = max(_MIN_MOSAIC_BLOCK_PX, sel.get('granularity', 4))
            scale_down = f'scale=iw/{block}:ih/{block}:flags=neighbor'
            scale_up   = f'scale=iw*{block}:ih*{block}:flags=neighbor'
            crops.append(
                f'[0:v]crop={cw}:{ch}:{x0}:{y0},{scale_down},{scale_up}[mz{i}]'
            )
        overlays.append(
            f"{prev}[mz{i}]overlay={x0}:{y0}:enable='between(t,{t_start:.6f},{t_end:.6f})'{label}"
        )
        prev = f'[tmp{i}]'
    return ';'.join(crops + overlays)

state = {
    'source': None,
    'duration': 0.0,
    'fps': 30.0,
    'orig_w': 0,
    'orig_h': 0,
    'frame_sec': 0.0,
    'selections': [],   # [{x, y, w, h, granularity, pattern, color}, ...]
    'add_mode': True,
    'preview_hidden': True,
    'last_output': None,
}

print('✅ セットアップ完了')


In [ ]:
# @title 2. 動画アップロード・モザイク指定 { display-mode: "form" }
# @markdown 動画をアップロードし、モザイクをかけるフレームと位置を指定します。
# @markdown 1. 動画ファイルを選択してアップロード
# @markdown 2. 再生して対象フレームで「開始秒を記録」「終了秒を記録」をクリック
# @markdown 3. ＋ ボタンをONにしてフレーム画像をクリックしモザイク位置を指定


DISPLAY_W = 720

def _extract_frame(sec):
    _ffmpeg('-ss', f'{sec:.4f}', '-i', state['source'],
            '-frames:v', '1', '-q:v', '2', 'current_frame.png')

def _apply_mosaic_pil(img, sel):
    x, y, sw, sh = sel['x'], sel['y'], sel['w'], sel['h']
    x0, y0 = max(0, x - sw // 2), max(0, y - sh // 2)
    x1, y1 = min(img.width, x + sw // 2), min(img.height, y + sh // 2)
    if x1 <= x0 or y1 <= y0:
        return img
    if sel.get('pattern') == 'solid':
        clr = sel.get('color', '#808080').lstrip('#')
        rgb = tuple(int(clr[i:i+2], 16) for i in (0, 2, 4))
        img.paste(rgb, (x0, y0, x1, y1))
    else:
        block  = max(_MIN_MOSAIC_BLOCK_PX, sel.get('granularity', 4))
        region = img.crop((x0, y0, x1, y1))
        rw, rh = region.size
        small  = region.resize((max(1, rw // block), max(1, rh // block)), PILImage.NEAREST)
        mosaic = small.resize((rw, rh), PILImage.NEAREST)
        img.paste(mosaic, (x0, y0))
    return img

def _render_preview_b64():
    img = PILImage.open('current_frame.png').convert('RGB')
    if not state['preview_hidden']:
        for sel in state['selections']:
            _apply_mosaic_pil(img, sel)
    img     = img.convert('RGBA')
    overlay = PILImage.new('RGBA', img.size, (0, 0, 0, 0))
    draw    = ImageDraw.Draw(overlay)
    for sel in state['selections']:
        x, y, sw, sh = sel['x'], sel['y'], sel['w'], sel['h']
        x0, y0, x1, y1 = x - sw // 2, y - sh // 2, x + sw // 2, y + sh // 2
        if sel.get('pattern') == 'solid':
            clr = sel.get('color', '#808080').lstrip('#')
            outline_color = tuple(int(clr[i:i+2], 16) for i in (0, 2, 4)) + (230,)
        else:
            outline_color = (255, 50, 50, 230)
        draw.rectangle((x0, y0, x1, y1), outline=outline_color, width=3)
        draw.ellipse((x - 4, y - 4, x + 4, y + 4), fill=(255, 50, 50, 200))
        draw.rectangle((x1 - 8, y1 - 8, x1 + 8, y1 + 8), fill=(0, 210, 255, 240), outline=(255, 255, 255, 255), width=2)
    combined = PILImage.alpha_composite(img, overlay).convert('RGB')
    buf = BytesIO()
    combined.save(buf, format='JPEG', quality=_PREVIEW_JPEG_QUALITY)
    return base64.b64encode(buf.getvalue()).decode()

def _build_canvas_html(b64, add_mode=False):
    disp_h    = int(DISPLAY_W * state['orig_h'] / state['orig_w'])
    scale_val = state['orig_w']
    add_mode_js = 'true' if add_mode else 'false'
    sels_js = json.dumps(state['selections']).replace('"', '&quot;')
    return f"""
<div style="position:relative;display:inline-block">
  <img src="data:image/jpeg;base64,{b64}" width="{DISPLAY_W}" style="display:block">
  <canvas id="mz_cv" data-sels="{sels_js}" width="{DISPLAY_W}" height="{disp_h}"
    style="position:absolute;top:0;left:0;width:100%;height:100%;cursor:{'crosshair' if add_mode else 'default'}"
    onmousedown="(function(e){{
      var cv=e.currentTarget;
      var addMode={add_mode_js};
      var r=cv.getBoundingClientRect();
      var sc={scale_val}/r.width;
      var mx=Math.round((e.clientX-r.left)*sc);
      var my=Math.round((e.clientY-r.top)*sc);
      var sels={sels_js};
      var HANDLE=Math.round(16*sc);
      window._mz={{mode:null,idx:-1,ds:{{x:mx,y:my}},ox:0,oy:0,cv:cv,sc:sc}};
      for(var i=sels.length-1;i>=0;i--){{
        var s=sels[i];
        var hx=s.x+s.w/2, hy=s.y+s.h/2;
        if(Math.abs(mx-hx)<=HANDLE&&Math.abs(my-hy)<=HANDLE){{
          window._mz.mode='resize'; window._mz.idx=i; break;
        }}
      }}
      if(!window._mz.mode){{
        for(var i=sels.length-1;i>=0;i--){{
          var s=sels[i];
          if(mx>=s.x-s.w/2&&mx<=s.x+s.w/2&&my>=s.y-s.h/2&&my<=s.y+s.h/2){{
            window._mz.mode='move'; window._mz.idx=i;
            window._mz.ox=mx-s.x; window._mz.oy=my-s.y;
            break;
          }}
        }}
      }}
      function _onUp(ev){{
        document.removeEventListener('mouseup',_onUp);
        if(!window._mz) return;
        var mz=window._mz;
        var r2=mz.cv.getBoundingClientRect();
        var ex=Math.round((ev.clientX-r2.left)*mz.sc);
        var ey=Math.round((ev.clientY-r2.top)*mz.sc);
        var moved=Math.abs(ex-mz.ds.x)>3||Math.abs(ey-mz.ds.y)>3;
        if(mz.mode==='resize'){{
          google.colab.kernel.invokeFunction('mz_resize',[mz.idx,ex,ey],{{}});
        }}else if(mz.mode==='move'){{
          google.colab.kernel.invokeFunction('mz_move',[mz.idx,ex-mz.ox,ey-mz.oy],{{}});
        }}else if(!mz.mode&&!moved&&addMode){{
          google.colab.kernel.invokeFunction('mz_click',[ex,ey],{{}});
        }}
        window._mz=null;
      }}
      document.addEventListener('mouseup',_onUp);
      e.preventDefault();
    }})(event)"
    onmousemove="(function(e){{
      var cv=e.currentTarget;
      if(window._mz&&window._mz.mode){{cv.style.cursor=window._mz.mode==='resize'?'nw-resize':'grabbing';return;}}
      var r=cv.getBoundingClientRect();
      var sc={scale_val}/r.width;
      var mx=Math.round((e.clientX-r.left)*sc);
      var my=Math.round((e.clientY-r.top)*sc);
      var sels=JSON.parse((cv.dataset.sels||'[]').replace(/&quot;/g,'"'));
      var HANDLE=Math.round(16*sc);
      var addMode={add_mode_js};
      var cur=addMode?'crosshair':'default';
      for(var i=sels.length-1;i>=0;i--){{var s=sels[i];
        if(Math.abs(mx-(s.x+s.w/2))<=HANDLE&&Math.abs(my-(s.y+s.h/2))<=HANDLE){{cur='nw-resize';break;}}
        if(mx>=s.x-s.w/2&&mx<=s.x+s.w/2&&my>=s.y-s.h/2&&my<=s.y+s.h/2){{cur='grab';break;}}
      }}
      cv.style.cursor=cur;
    }})(event)">
  </canvas>
</div>
<div>
  {'🖱 枠内ドラッグ: 移動 ／ 右下□ドラッグ: リサイズ' if not add_mode else '✚ クリック: 新規追加（1クリックで追加モードOFF） ／ 枠内ドラッグ: 移動 ／ 右下□ドラッグ: リサイズ'}
</div>
"""

status_out = w.Output()
frame_out  = w.Output()
sel_label  = w.Label('📋 選択: 0 箇所')

range_start = w.BoundedFloatText(value=0, min=0, max=9999, step=0.01,
                                  description='開始(秒):',
                                  layout=w.Layout(width='200px'),
                                  style={'description_width': '70px'})
range_end   = w.BoundedFloatText(value=0, min=0, max=9999, step=0.01,
                                  description='終了(秒):',
                                  layout=w.Layout(width='200px'),
                                  style={'description_width': '70px'})
range_hint  = w.Label('（開始 = 終了 のとき1フレームのみ）')

size_w_slider = w.IntSlider(value=80, min=10, max=600, step=10,
                             description='幅(px):',
                             layout=w.Layout(width='380px'),
                             style={'description_width': '60px'})
size_h_slider = w.IntSlider(value=80, min=10, max=600, step=10,
                             description='高さ(px):',
                             layout=w.Layout(width='380px'),
                             style={'description_width': '60px'})
gran_slider = w.IntSlider(value=4, min=2, max=40, step=1,
                           description='粒度:',
                           layout=w.Layout(width='260px'),
                           style={'description_width': '60px'})
gran_hint  = w.Label('小 → 細かいモザイク ／ 大 → 粗いモザイク')

pattern_sel = w.RadioButtons(
    options=[('モザイク', 'mosaic'), ('単色', 'solid')],
    value='mosaic',
    description='パターン:',
    style={'description_width': '70px'},
    layout=w.Layout(width='200px'),
)
color_picker = w.ColorPicker(
    value='#cdcdcd',
    description='色:',
    layout=w.Layout(width='180px', display='none'),
    style={'description_width': '30px'},
)

def _on_pattern_change(change):
    is_solid = change['new'] == 'solid'
    color_picker.layout.display = '' if is_solid else 'none'
    gran_slider.layout.display  = 'none' if is_solid else ''
    gran_hint.layout.display    = 'none' if is_solid else ''

pattern_sel.observe(_on_pattern_change, names='value')

add_btn = w.ToggleButton(
    value=True,
    description='➕ モザイク追加 ON',
    button_style='warning',
    icon='plus',
    layout=w.Layout(width='160px'),
)

undo_btn  = w.Button(description='↩ 1つ戻す', button_style='warning',
                      layout=w.Layout(width='110px'))
clear_btn = w.Button(description='🗑 全消去', button_style='danger',
                      layout=w.Layout(width='100px'))

_slider_sync = False  # スライダー同期中は _on_setting_change をスキップ

def _refresh_frame_display():
    b64 = _render_preview_b64()
    with frame_out:
        frame_out.clear_output(wait=True)
        display(HTML(_build_canvas_html(b64, add_mode=state['add_mode'])))
    sel_label.value = f'📋 選択: {len(state["selections"])} 箇所'

def _on_show_frame(sec):
    state['frame_sec'] = sec
    _extract_frame(sec)
    _refresh_frame_display()

def mz_click(rx, ry):
    if not state['add_mode']:
        return
    state['selections'].append({
        'x': rx, 'y': ry,
        'w': size_w_slider.value,
        'h': size_h_slider.value,
        'granularity': gran_slider.value,
        'pattern': pattern_sel.value,
        'color': color_picker.value,
    })
    state['add_mode'] = False
    add_btn.value = False
    _refresh_frame_display()

colab_output.register_callback('mz_click', mz_click)

def mz_move(idx, new_x, new_y):
    if 0 <= idx < len(state['selections']):
        state['selections'][idx]['x'] = new_x
        state['selections'][idx]['y'] = new_y
        _refresh_frame_display()

colab_output.register_callback('mz_move', mz_move)

def mz_resize(idx, end_x, end_y):
    if 0 <= idx < len(state['selections']):
        sel = state['selections'][idx]
        new_w = max(_MIN_SELECTION_SIZE, abs(end_x - sel['x']) * 2)
        new_h = max(_MIN_SELECTION_SIZE, abs(end_y - sel['y']) * 2)
        sel['w'] = int(new_w)
        sel['h'] = int(new_h)
        global _slider_sync
        _slider_sync = True
        size_w_slider.value = min(size_w_slider.max, sel['w'])
        size_h_slider.value = min(size_h_slider.max, sel['h'])
        _slider_sync = False
        _refresh_frame_display()

colab_output.register_callback('mz_resize', mz_resize)

def _on_add_mode_toggle(change):
    state['add_mode'] = change['new']
    add_btn.description = '➕ モザイク追加 ON' if change['new'] else '➕ モザイク追加 OFF'
    add_btn.button_style = 'warning' if change['new'] else ''
    if os.path.exists('current_frame.png'):
        _refresh_frame_display()

add_btn.observe(_on_add_mode_toggle, names='value')

def _on_undo(_):
    if state['selections']:
        state['selections'].pop()
        _refresh_frame_display()

def _on_clear(_):
    state['selections'].clear()
    _refresh_frame_display()

undo_btn.on_click(_on_undo)
clear_btn.on_click(_on_clear)

peek_btn = w.ToggleButton(
    value=False,
    description='👁 配色を確認',
    button_style='',
    layout=w.Layout(width='120px'),
)

def _on_peek_toggle(change):
    state['preview_hidden'] = not change['new']
    peek_btn.button_style = 'info' if change['new'] else ''
    if os.path.exists('current_frame.png'):
        _refresh_frame_display()

peek_btn.observe(_on_peek_toggle, names='value')

def _on_setting_change(change):
    if _slider_sync or not state['selections'] or not os.path.exists('current_frame.png'):
        return
    sel = state['selections'][-1]
    sel['w']           = size_w_slider.value
    sel['h']           = size_h_slider.value
    sel['granularity'] = gran_slider.value
    sel['pattern']     = pattern_sel.value
    sel['color']       = color_picker.value
    _refresh_frame_display()

for _widget in (size_w_slider, size_h_slider, gran_slider, pattern_sel, color_picker):
    _widget.observe(_on_setting_change, names='value')

print('📂 動画ファイルを選択してください（対応形式: .mov .mp4 .avi .mkv .webm）')
uploaded = colab_files.upload()

VIDEO_EXT = ('.mov', '.mp4', '.avi', '.mkv', '.webm')
videos = [f for f in uploaded.keys() if f.lower().endswith(VIDEO_EXT)]
if not videos:
    print('❌ 動画ファイルが見つかりません')
else:
    src = videos[0]
    state['source'] = src

    state['duration'] = _safe_float(_probe(src, 'duration'))

    res = subprocess.run(
        ['ffprobe', '-v', 'error', '-select_streams', 'v:0',
         '-show_entries', 'stream=width,height,r_frame_rate',
         '-of', 'csv=p=0', src],
        capture_output=True, text=True
    )
    parts = res.stdout.strip().split(',')
    if len(parts) < 3:
        raise RuntimeError(f'ffprobe が動画情報を取得できませんでした: {res.stdout!r}')
    state['orig_w'] = int(parts[0])
    state['orig_h'] = int(parts[1])
    try:
        num, den = parts[2].split('/')
        state['fps'] = float(num) / float(den) if float(den) != 0 else 30.0
    except (ValueError, ZeroDivisionError):
        state['fps'] = 30.0

    dur = state['duration']
    range_start.max = dur
    range_end.max   = dur

    size_mb = os.path.getsize(src) / 1024 / 1024
    print(f'\n✅ {src} ({size_mb:.1f} MB)')
    print(f'   解像度: {state["orig_w"]}x{state["orig_h"]} / FPS: {state["fps"]:.2f} / 長さ: {dur:.1f}秒')
    print('\n▶ 動画を再生して位置を確認し、一時停止したフレームにモザイクを指定してください')

    video_container = w.Output()

    def _reload_video_widget():
        with open(state['source'], 'rb') as _vf:
            _b64v = base64.b64encode(_vf.read()).decode()
        ext = state['source'].rsplit('.', 1)[-1].lower()
        mime = 'video/mp4' if ext == 'mp4' else f'video/{ext}'
        video_html = w.HTML(f"""
<div>
  <video id="mz_video" width="720" controls>
    <source src="data:{mime};base64,{_b64v}" type="{mime}">
  </video>
  <div style="display:flex;gap:8px;margin-top:4px;align-items:center">
    <button
      onclick="(function(){{var v=document.getElementById('mz_video');google.colab.kernel.invokeFunction('mz_set_start',[v.currentTime],{{}});}})()">📍 開始秒を記録
    </button>
    <button
      onclick="(function(){{var v=document.getElementById('mz_video');google.colab.kernel.invokeFunction('mz_set_end',[v.currentTime],{{}});}})()">📍 終了秒を記録
    </button>
    <span style="color:white;font-size:12px;line-height:1.6">📍 開始秒を記録 を押した時点の映像フレームが下部の編集画面に反映されます</span>
  </div>
</div>
""")
        with video_container:
            video_container.clear_output(wait=True)
            display(video_html)

    def mz_set_start(t):
        range_start.value = round(float(t), 2)
        sec = round(float(t), 4)
        state['frame_sec'] = sec
        _on_show_frame(sec)

    def mz_set_end(t):
        range_end.value = round(float(t), 2)

    colab_output.register_callback('mz_set_start', mz_set_start)
    colab_output.register_callback('mz_set_end', mz_set_end)

    _reload_video_widget()

    display(w.VBox([
        video_container,
        status_out,
        w.Label('🕒 モザイク適用範囲'),
        w.HBox([range_start, range_end, range_hint]),
        w.Label('🖱 モザイク設定'),
        w.HBox([size_w_slider, size_h_slider]),
        w.HBox([pattern_sel, color_picker, w.VBox([gran_slider, gran_hint])]),
        w.HBox([add_btn, undo_btn, clear_btn, peek_btn, sel_label],
               layout=w.Layout(gap='8px', align_items='center')),
        frame_out,
    ]))

    _on_show_frame(0.0)


In [ ]:
# @title 3. 書き出し { display-mode: "form" }
# @markdown モザイクを適用した動画を MP4 (H.264) で書き出します。
# @markdown
# @markdown 出力横幅・保存先を設定して`▶書き出し実行`を押してください。


if not state['source']:
    raise RuntimeError('❌ 動画が読み込まれていません。② を先に実行してください')
if not os.path.exists('current_frame.png'):
    raise RuntimeError('❌ フレームが抽出されていません。② でフレームを表示してください')
if not state['selections']:
    raise RuntimeError('❌ モザイク位置が選択されていません。② でクリックしてください')

mp4_scale = w.BoundedIntText(
    value=960, min=240, max=1920,
    description='横幅 px:',
    layout=w.Layout(width='220px'),
    style={'description_width': '70px'}
)

save_sel = w.RadioButtons(
    options=[
        ('ローカルにダウンロード', 'local'),
        ('Google Drive に保存',   'drive'),
        ('両方',                  'both'),
    ],
    value='local', description='保存先:',
    style={'description_width': '60px'}
)
drive_path = w.Text(
    value='MyDrive/output_mosaic',
    description='Drive パス:',
    layout=w.Layout(width='380px', display='none'),
    style={'description_width': '80px'}
)

def _on_save_change(change):
    drive_path.layout.display = '' if change['new'] in ('drive', 'both') else 'none'

save_sel.observe(_on_save_change, names='value')

run_btn  = w.Button(description='▶ 書き出し実行', button_style='success',
                    layout=w.Layout(width='150px'))
out_log  = w.Output()
display(w.VBox([
    w.Label('📦 出力形式: MP4 (H.264)'),
    mp4_scale,
    w.Label('💾 保存先'),
    save_sel,
    drive_path,
    run_btn,
    out_log,
]))

def _run_export():
    output_name = 'output_mosaic.mp4'
    t_start     = range_start.value
    t_end       = range_end.value

    if t_end < t_start:
        t_end = t_start
    if t_end == t_start:
        t_end = t_start + 1.0 / state['fps']

    src   = state['source']
    scale = mp4_scale.value

    fc = _build_filter_complex(
        state['selections'], t_start, t_end,
        orig_w=state['orig_w'], orig_h=state['orig_h'], out='[vr]'
    )
    fc += f';[vr]scale=trunc({scale}/2)*2:-2:flags=lanczos[vout]'

    with out_log:
        print(f'[1/1] MP4 生成中 (モザイク範囲: {t_start:.2f}秒 〜 {t_end:.2f}秒)...')
        _ffmpeg_progress([
            '-i', src,
            '-filter_complex', fc,
            '-map', '[vout]',
            '-c:v', 'libx264', '-crf', '23', '-preset', 'slow',
            '-pix_fmt', 'yuv420p',
            '-map', '0:a?',
            '-c:a', 'aac',
            output_name
        ], label='MP4エンコード')

    size_mb = os.path.getsize(output_name) / 1024 / 1024
    with out_log:
        print(f'\n✅ 書き出し完了 → {output_name} ({size_mb:.1f} MB)')
        print('\n▶ プレビュー:')
        display(Video(output_name, embed=True, width=720))

    state['last_output'] = output_name

def _on_run(_):
    run_btn.disabled = True
    with out_log:
        out_log.clear_output(wait=True)
    try:
        _run_export()
    except Exception as e:
        import traceback
        with out_log:
            print(f'❌ エラー: {e}')
            traceback.print_exc()
    run_btn.disabled = False

run_btn.on_click(_on_run)


In [ ]:
# @title 4. 保存 / ダウンロード { display-mode: "form" }
# @markdown 書き出した MP4 をローカルまたは Google Drive に保存します。


if not state.get('last_output') or not os.path.exists(state['last_output']):
    raise RuntimeError('❌ 書き出しファイルが見つかりません。③ を先に実行してください')

output_name = state['last_output']
method      = save_sel.value

if method in ('local', 'both'):
    print(f'💾 ダウンロード開始: {output_name}')
    colab_files.download(output_name)
    print('✅ ローカルにダウンロードしました')

if method in ('drive', 'both'):
    dp = drive_path.value.strip().strip('/')
    if not dp:
        raise RuntimeError('❌ Drive パスが空です。drive_path に保存先パスを入力してください')
    if not dp.endswith('.mp4'):
        dp = dp + '.mp4'
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    dest = '/content/drive/' + dp
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    shutil.copy(output_name, dest)
    print(f'✅ Drive に保存しました → {dp}')
